In [1]:
# ============================================================
# 05_baseline_rgb.ipynb
# STABLE RGB BASELINE - H100 & NFS OPTIMIZED (Jupyter Safe)
# Uses frame_extraction_trimmed_summary.csv
# ============================================================

import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

TRAIN_CSV_NAME = "protocol_A_train.csv"
VAL_CSV_NAME   = "protocol_A_val.csv"
TEST_CSV_NAME  = "protocol_A_test.csv"

FRAME_SOURCE = "RGB"

MODELS_TO_RUN = ["resnet18"]

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 8

BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4
USE_AMP = True
USE_PRETRAINED = True

# OPTIMIZED for Jupyter Notebooks (avoids multiprocessing AttributeError)
NUM_WORKERS = 0      
PIN_MEMORY = False    

LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

SAMPLE_START_FRAC = 0.15
SAMPLE_END_FRAC = 0.85

# ============================================================
# 2. SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
RESULTS_ROOT = ROOT / "results"
CHECKPOINT_ROOT = ROOT / "checkpoints"
LOG_ROOT = ROOT / "logs"

TRAIN_CSV = SPLIT_DIR / TRAIN_CSV_NAME
VAL_CSV = SPLIT_DIR / VAL_CSV_NAME
TEST_CSV = SPLIT_DIR / TEST_CSV_NAME
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, FRAME_SUMMARY_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

print("ROOT:", ROOT)

# ============================================================
# 4. LOAD SPLITS
# ============================================================
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

if "is_valid_pair" in train_df.columns:
    train_df = train_df[train_df["is_valid_pair"] == True].copy()
if "is_valid_pair" in val_df.columns:
    val_df = val_df[val_df["is_valid_pair"] == True].copy()
if "is_valid_pair" in test_df.columns:
    test_df = test_df[test_df["is_valid_pair"] == True].copy()

# ============================================================
# 5. LABELS
# ============================================================
CLASS_NAMES = sorted(train_df["gesture"].dropna().unique().tolist())
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

for df_ in [train_df, val_df, test_df]:
    df_["label"] = df_["gesture"].map(label2id)

# ============================================================
# 6. LOAD FRAME SUMMARY
# ============================================================
frame_df = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df = frame_df[
    (frame_df["modality"] == FRAME_SOURCE) &
    (frame_df["status"] == "success")
].copy()

frame_df = frame_df[["pair_id", "output_dir", "frames_extracted"]].drop_duplicates()
frame_df = frame_df.rename(columns={
    "output_dir": "frame_dir",
    "frames_extracted": "num_frames_available"
})

def attach_frame_info(df_):
    out = df_.merge(frame_df, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

train_df = attach_frame_info(train_df)
val_df = attach_frame_info(val_df)
test_df = attach_frame_info(test_df)

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ============================================================
# 8. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.15, end_frac=0.85):
    if n_total <= 0: return [0] * n_samples
    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))
    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))
    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

# ============================================================
# 9. DATASET
# ============================================================
class FastFrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=8, start_frac=0.15, end_frac=0.85):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.num_sample_frames = num_sample_frames
        self.sampled_paths = []
        
        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames, start_frac, end_frac
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]
        frames = []
        
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        
        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, int(row["label"]), meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

print("\nBuilding dataloaders...")
train_loader = DataLoader(
    FastFrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
)
val_loader = DataLoader(
    FastFrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
)
test_loader = DataLoader(
    FastFrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
)

# ============================================================
# 10. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Fixed GPU lock issue for H100s
torch.backends.cudnn.benchmark = False 

# ============================================================
# 11. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    return model

# ============================================================
# 12. FORWARD
# ============================================================
def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W) 
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    return frame_logits.mean(dim=1)

# ============================================================
# 13. TRAIN / EVAL 
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []
    total_batches = len(loader)

    for batch_idx, (frames, labels, metas) in enumerate(loader):
        frames, labels = frames.to(device), labels.to(device)

        if train: optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        if batch_idx % 10 == 0 or batch_idx == total_batches - 1:
            mode = "Train" if train else "Val"
            print(f"   --> {mode} Batch {batch_idx}/{total_batches} | Current Loss: {loss.item():.4f}", flush=True)

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro")
    }

# ============================================================
# 14. RUN EXPERIMENT
# ============================================================
def run_experiment(model_name):
    print("\n" + "=" * 80)
    print(f"RUNNING MODEL: {model_name}")
    
    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1, best_epoch = -1.0, -1
    
    for epoch in range(1, EPOCHS + 1):
        print(f"\n[{model_name}] Epoch {epoch}/{EPOCHS}")
        
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        print(f"   Train Loss: {train_metrics['loss']:.4f} | Val Loss: {val_metrics['loss']:.4f}")
        print(f"   Train F1:   {train_metrics['macro_f1']:.4f} | Val F1:   {val_metrics['macro_f1']:.4f}")

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            print(f"   * New best model saved *")

    return best_val_f1

for model_name in MODELS_TO_RUN:
    run_experiment(model_name)
    
print("\nFINISHED!")

ROOT: /data/Sajjan_Singh/gesture_recognition

Building dataloaders...

Using device: cuda
GPU name: NVIDIA H100 NVL

RUNNING MODEL: resnet18

[resnet18] Epoch 1/10
   --> Train Batch 0/66 | Current Loss: 2.5154
   --> Train Batch 10/66 | Current Loss: 2.5040
   --> Train Batch 20/66 | Current Loss: 1.7203
   --> Train Batch 30/66 | Current Loss: 1.7174
   --> Train Batch 40/66 | Current Loss: 1.2775
   --> Train Batch 50/66 | Current Loss: 1.5070
   --> Train Batch 60/66 | Current Loss: 1.5888
   --> Train Batch 65/66 | Current Loss: 1.4099
   --> Val Batch 0/10 | Current Loss: 0.5421
   --> Val Batch 9/10 | Current Loss: 0.4160
   Train Loss: 1.7261 | Val Loss: 0.8938
   Train F1:   0.2718 | Val F1:   0.5401
   * New best model saved *

[resnet18] Epoch 2/10
   --> Train Batch 0/66 | Current Loss: 1.0445
   --> Train Batch 10/66 | Current Loss: 0.7132
   --> Train Batch 20/66 | Current Loss: 0.7819
   --> Train Batch 30/66 | Current Loss: 0.8245
   --> Train Batch 40/66 | Current Loss

In [1]:
# ============================================================
# 05_baseline_rgb.ipynb
# FULL RGB BASELINE FOR ALL 3 MODELS
# Saves:
# - best checkpoint
# - training log CSV
# - test predictions CSV
# - classification report CSV
# - confusion matrix PNG
# - metrics JSON
# - final comparison CSV
# ============================================================

import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

TRAIN_CSV_NAME = "protocol_A_train.csv"
VAL_CSV_NAME   = "protocol_A_val.csv"
TEST_CSV_NAME  = "protocol_A_test.csv"

# "RGB" or "RGB_BGREM"
FRAME_SOURCE = "RGB"

MODELS_TO_RUN = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_small",
]

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 8

BATCH_SIZE = 8
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4
USE_AMP = True
USE_PRETRAINED = True

# Jupyter-safe
NUM_WORKERS = 0
PIN_MEMORY = False

LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

SAMPLE_START_FRAC = 0.15
SAMPLE_END_FRAC = 0.85

# ============================================================
# 2. SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
RESULTS_ROOT = ROOT / "results"
CHECKPOINT_ROOT = ROOT / "checkpoints"
LOG_ROOT = ROOT / "logs"

TRAIN_CSV = SPLIT_DIR / TRAIN_CSV_NAME
VAL_CSV = SPLIT_DIR / VAL_CSV_NAME
TEST_CSV = SPLIT_DIR / TEST_CSV_NAME
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, FRAME_SUMMARY_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

print("ROOT:", ROOT)
print("FRAME_SOURCE:", FRAME_SOURCE)
print("TRAIN_CSV:", TRAIN_CSV)
print("VAL_CSV:", VAL_CSV)
print("TEST_CSV:", TEST_CSV)
print("FRAME_SUMMARY_CSV:", FRAME_SUMMARY_CSV)

# ============================================================
# 4. LOAD SPLITS
# ============================================================
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

if "is_valid_pair" in train_df.columns:
    train_df = train_df[train_df["is_valid_pair"] == True].copy()
if "is_valid_pair" in val_df.columns:
    val_df = val_df[val_df["is_valid_pair"] == True].copy()
if "is_valid_pair" in test_df.columns:
    test_df = test_df[test_df["is_valid_pair"] == True].copy()

if LIMIT_TRAIN is not None:
    train_df = train_df.head(LIMIT_TRAIN).copy()
if LIMIT_VAL is not None:
    val_df = val_df.head(LIMIT_VAL).copy()
if LIMIT_TEST is not None:
    test_df = test_df.head(LIMIT_TEST).copy()

print("\nTrain shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

# ============================================================
# 5. LABELS
# ============================================================
CLASS_NAMES = sorted(train_df["gesture"].dropna().unique().tolist())
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

for df_ in [train_df, val_df, test_df]:
    df_["label"] = df_["gesture"].map(label2id)

print("\nCLASS_NAMES:", CLASS_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)

# ============================================================
# 6. LOAD FRAME SUMMARY
# ============================================================
frame_df = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df = frame_df[
    (frame_df["modality"] == FRAME_SOURCE) &
    (frame_df["status"] == "success")
].copy()

frame_df = frame_df[["pair_id", "output_dir", "frames_extracted"]].drop_duplicates()
frame_df = frame_df.rename(columns={
    "output_dir": "frame_dir",
    "frames_extracted": "num_frames_available"
})

def attach_frame_info(df_):
    out = df_.merge(frame_df, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

train_df = attach_frame_info(train_df)
val_df = attach_frame_info(val_df)
test_df = attach_frame_info(test_df)

print("\nAfter merging frame summary:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ============================================================
# 8. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.15, end_frac=0.85):
    if n_total <= 0:
        return [0] * n_samples

    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))

    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))

    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

# ============================================================
# 9. DATASET
# ============================================================
class FastFrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=8, start_frac=0.15, end_frac=0.85):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.num_sample_frames = num_sample_frames
        self.sampled_paths = []

        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                start_frac,
                end_frac
            )
            self.sampled_paths.append(
                [frame_index_to_path(row["frame_dir"], i) for i in idxs]
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]
        frames = []

        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }

        return frames, int(row["label"]), meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

print("\nBuilding dataloaders...")
train_loader = DataLoader(
    FastFrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    FastFrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
test_loader = DataLoader(
    FastFrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)

# ============================================================
# 10. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = False

# ============================================================
# 11. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model

# ============================================================
# 12. FORWARD
# ============================================================
def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W)
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    return frame_logits.mean(dim=1)

# ============================================================
# 13. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro")
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()
    all_labels, all_preds, rows = [], [], []

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
            video_logits = forward_video_voting(model, frames)
            probs = torch.softmax(video_logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    return all_labels, all_preds, pd.DataFrame(rows)

# ============================================================
# 14. RUN EXPERIMENT
# ============================================================
def run_experiment(model_name):
    print("\n" + "=" * 80)
    print(f"RUNNING MODEL: {model_name}")

    run_name = f"rgb_trimmed_{FRAME_SOURCE.lower()}_{model_name}_f{NUM_SAMPLE_FRAMES}_img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    best_ckpt_path = checkpoint_dir / "best_model.pt"
    history = []

    for epoch in range(1, EPOCHS + 1):
        print(f"\n[{model_name}] Epoch {epoch}/{EPOCHS}")

        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })

        print(f"Train Loss: {train_metrics['loss']:.4f} | Val Loss: {val_metrics['loss']:.4f}")
        print(f"Train F1:   {train_metrics['macro_f1']:.4f} | Val F1:   {val_metrics['macro_f1']:.4f}")

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "model_name": model_name,
                "frame_source": FRAME_SOURCE,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
            print("   * New best model saved *")

    # Save training log
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    # Load best model and evaluate on test
    best_ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(best_ckpt["model_state_dict"])

    test_labels, test_preds, pred_df = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    cm = confusion_matrix(test_labels, test_preds)

    report_dict = classification_report(
        test_labels,
        test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"
    metrics_json = results_dir / "metrics.json"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix - {model_name} - {FRAME_SOURCE}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    metrics = {
        "frame_source": FRAME_SOURCE,
        "model_name": model_name,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusion_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(f"\n[{model_name}] FINAL RESULTS")
    print(json.dumps(metrics, indent=4))

    return metrics

# ============================================================
# 15. RUN ALL MODELS
# ============================================================
all_metrics = []

for model_name in MODELS_TO_RUN:
    metrics = run_experiment(model_name)
    all_metrics.append(metrics)

summary_df = pd.DataFrame(all_metrics)
summary_csv = RESULTS_ROOT / f"rgb_trimmed_{FRAME_SOURCE.lower()}_comparison.csv"
summary_df.to_csv(summary_csv, index=False)

print("\nFINISHED!")
print(summary_df)

ROOT: /data/Sajjan_Singh/gesture_recognition
FRAME_SOURCE: RGB
TRAIN_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_A_train.csv
VAL_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_A_val.csv
TEST_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_A_test.csv
FRAME_SUMMARY_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/frame_extraction_trimmed_summary.csv

Train shape: (525, 26)
Val shape  : (77, 26)
Test shape : (147, 26)

CLASS_NAMES: ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']
NUM_CLASSES: 7

After merging frame summary:
Train: (525, 29)
Val  : (77, 29)
Test : (147, 29)

Building dataloaders...

Using device: cuda
GPU name: NVIDIA H100 NVL

RUNNING MODEL: resnet18

[resnet18] Epoch 1/12


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 05_stage5_master_rgb.ipynb
# FINAL STAGE 5 RGB BENCHMARK
# ============================================================
# Covers:
#   Stage 5A: RGB frame baseline
#   Stage 5B: raw RGB vs RGB_BGREM ablation
#   Stage 5C: Protocol A + Protocol B cross-distance evaluation
#
# Models:
#   - resnet18
#   - efficientnet_b0
#   - mobilenet_v3_small
#
# Frame sources:
#   - RGB
#   - RGB_BGREM
#
# Splits:
#   - protocol_A
#   - 4 -> 8
#   - 8 -> 4
#   - 4+6 -> 8
#   - 4+8 -> 6
#   - 6+8 -> 4
#
# Metrics:
#   - accuracy
#   - macro_f1
#   - weighted_f1
#   - per_class_f1
#   - confusion_matrix
#   - train_time_minutes
#   - inference_time_seconds
#   - videos_per_second
# ============================================================

import json
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

BATCH_SIZE = 8
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True

# Jupyter-safe / stable
NUM_WORKERS = 0
PIN_MEMORY = False

# Optional quick test limits
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

# Smart temporal sampling from middle region
SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

# Run both raw RGB and bg-removed RGB
FRAME_SOURCES = [
    "RGB",
    "RGB_BGREM",
]

MODELS_TO_RUN = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_small",
]

# protocol_name, train_csv, val_csv, test_csv
# If val_csv is None, a small val split will be created from train.
SPLIT_CONFIGS = [
    ("protocol_A",        "protocol_A_train.csv",        "protocol_A_val.csv",        "protocol_A_test.csv"),
    ("protocol_B_4_to_8", "protocol_B_train_4_test_8.csv",   None,                     "protocol_B_test_8.csv"),
    ("protocol_B_8_to_4", "protocol_B_train_8_test_4.csv",   None,                     "protocol_B_test_4.csv"),
    ("protocol_B_4_6_to_8", "protocol_B_train_4_6_test_8.csv", None,                   "protocol_B_test_8_from_4_6.csv"),
    ("protocol_B_4_8_to_6", "protocol_B_train_4_8_test_6.csv", None,                   "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_6_8_to_4", "protocol_B_train_6_8_test_4.csv", None,                   "protocol_B_test_4_from_6_8.csv"),
]

# Fixed class order
CLASS_NAMES = [
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

RESULTS_ROOT = ROOT / "results" / "stage5_rgb_master"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_rgb_master"
LOG_ROOT = ROOT / "logs" / "stage5_rgb_master"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"
if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing file: {FRAME_SUMMARY_CSV}")

print("ROOT:", ROOT)
print("FRAME_SUMMARY_CSV:", FRAME_SUMMARY_CSV)
print("CLASS_NAMES:", CLASS_NAMES)

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = False

# ============================================================
# 5. FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples

    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))

    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))

    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    # 1-based frame naming: frame_000001.jpg
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    out = out.reset_index(drop=True)
    return out

def attach_frame_info(df_, frame_source):
    sub = frame_df_all[frame_df_all["modality"] == frame_source].copy()
    sub = sub.rename(columns={
        "output_dir": "frame_dir",
        "frames_extracted": "num_frames_available"
    })
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()

    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ============================================================
# 8. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32, start_frac=0.10, end_frac=0.90):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.num_sample_frames = num_sample_frames
        self.sampled_paths = []

        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                start_frac,
                end_frac
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]

        frames = []
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        label = int(row["label"])

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }

        return frames, label, meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

# ============================================================
# 9. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model

def forward_video_voting(model, frames):
    # frames: [B, T, C, H, W]
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W)
    frame_logits = model(frames)            # [B*T, num_classes]
    frame_logits = frame_logits.view(B, T, -1)
    video_logits = frame_logits.mean(dim=1) # [B, num_classes]
    return video_logits

# ============================================================
# 10. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    rows = []

    start_time = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
            video_logits = forward_video_voting(model, frames)
            probs = torch.softmax(video_logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        total_samples += len(labels_np)

        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    elapsed = time.time() - start_time
    videos_per_sec = total_samples / elapsed if elapsed > 0 else None

    return all_labels, all_preds, pd.DataFrame(rows), elapsed, videos_per_sec

# ============================================================
# 11. RUN ONE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name):
    print("\n" + "=" * 100)
    print(f"RUNNING | protocol={protocol_name} | frame_source={frame_source} | model={model_name}")
    print("=" * 100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)

    # If no val file is given, create a small val split from train
    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    if LIMIT_TRAIN is not None:
        train_df = train_df.head(LIMIT_TRAIN).copy()
    if LIMIT_VAL is not None:
        val_df = val_df.head(LIMIT_VAL).copy()
    if LIMIT_TEST is not None:
        test_df = test_df.head(LIMIT_TEST).copy()

    train_df = attach_labels(train_df)
    val_df = attach_labels(val_df)
    test_df = attach_labels(test_df)

    train_df = attach_frame_info(train_df, frame_source)
    val_df = attach_frame_info(val_df, frame_source)
    test_df = attach_frame_info(test_df, frame_source)

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    train_loader = DataLoader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )

    run_name = f"{protocol_name}__{frame_source.lower()}__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    best_ckpt_path = checkpoint_dir / "best_model.pt"
    best_state_dict = None

    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "model_name": model_name,
                "frame_source": frame_source,
                "protocol_name": protocol_name,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_elapsed = time.time() - train_start
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    if best_state_dict is None:
        raise RuntimeError("No best model was saved.")

    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_elapsed, videos_per_sec = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    cm = confusion_matrix(test_labels, test_preds)
    report_dict = classification_report(
        test_labels,
        test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"
    metrics_json = results_dir / "metrics.json"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{protocol_name} | {frame_source} | {model_name}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "frame_source": frame_source,
        "model_name": model_name,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_elapsed / 60.0),
        "inference_time_seconds": float(infer_elapsed),
        "videos_per_second": float(videos_per_sec) if videos_per_sec is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusionCNN+LSTM / CNN+GRU

3D CNN baseline

SlowFast_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(json.dumps(metrics, indent=2))
    return metrics

# ============================================================
# 12. RUN ALL EXPERIMENTS
# ============================================================
all_metrics = []

for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv_name, val_csv_name, test_csv_name in SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            metrics = run_experiment(
                protocol_name=protocol_name,
                train_csv_name=train_csv_name,
                val_csv_name=val_csv_name,
                test_csv_name=test_csv_name,
                frame_source=frame_source,
                model_name=model_name
            )
            all_metrics.append(metrics)

summary_df = pd.DataFrame(all_metrics)
summary_csv = RESULTS_ROOT / "stage5_rgb_master_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\nDONE.")
print("Saved summary to:", summary_csv)
display(summary_df[[
    "protocol_name",
    "frame_source",
    "model_name",
    "best_val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
    "train_time_minutes",
    "videos_per_second"
]].sort_values(["protocol_name", "frame_source", "test_macro_f1"], ascending=[True, True, False]))

SyntaxError: unterminated string literal (detected at line 634) (2641802793.py, line 634)